In [326]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [327]:
import torch
from torch.autograd import Variable
import numpy as np
import torch.functional as F
import torch.nn.functional as F

In [328]:
corpus = [
    'he is a king',
    'she is a queen',
    'he is a man',
    'she is a woman',
    'warsaw is poland capital',
    'berlin is germany capital',
    'paris is france capital',   
]

In [329]:
def tokenize_corpus(corpus):
    tokens = [x.split() for x in corpus]
    return tokens

tokenized_corpus = tokenize_corpus(corpus)
print(tokenized_corpus)

[['he', 'is', 'a', 'king'], ['she', 'is', 'a', 'queen'], ['he', 'is', 'a', 'man'], ['she', 'is', 'a', 'woman'], ['warsaw', 'is', 'poland', 'capital'], ['berlin', 'is', 'germany', 'capital'], ['paris', 'is', 'france', 'capital']]


In [330]:
def gen_index(tokenized_corpus):
    vocabulary = []
    for sentence in tokenized_corpus:
        for token in sentence:
            if token not in vocabulary:
                vocabulary.append(token)

    word2idx = {w: idx for (idx, w) in enumerate(vocabulary)}
    idx2word = {idx: w for (idx, w) in enumerate(vocabulary)}

    vocabulary_size = len(vocabulary)
    return word2idx, idx2word, vocabulary_size
word2idx, idx2word, vocabulary_size = gen_index(tokenized_corpus)

In [331]:
def gen_index_pairs(tokenized_corpus, word2idx=word2idx, window_size=2):
    idx_pairs = []
    # for each sentence
    for sentence in tokenized_corpus:
        indices = [word2idx[word] for word in sentence]
        # for each word, threated as center word
        for center_word_pos in range(len(indices)):
            # for each window position
            for w in range(-window_size, window_size + 1):
                context_word_pos = center_word_pos + w
                # make soure not jump out sentence
                if context_word_pos < 0 or context_word_pos >= len(indices) or center_word_pos == context_word_pos:
                    continue
                context_word_idx = indices[context_word_pos]
                idx_pairs.append((indices[center_word_pos], context_word_idx))

    idx_pairs = np.array(idx_pairs) # it will be useful to have this as numpy array
    return idx_pairs
idx_pairs = gen_index_pairs(tokenized_corpus)

In [332]:
def get_input_layer(word_idx, vocabulary_size):
    x = torch.zeros(vocabulary_size).float()
    x[word_idx] = 1.0
    return x

In [333]:
embedding_dims, num_epochs = 5, 101 # This is the original
learning_rate = 0.001

def w2v_train(embedding_dims, num_epochs, learning_rate, vocabulary_size, idx_pairs=idx_pairs, print_loss=True):
    """Train a Word2Vec model using the Skip-Gram approach."""
    # Adding so that results are always the same
    torch.manual_seed(0)
    np.random.seed(0)
    W1 = Variable(torch.randn(embedding_dims, vocabulary_size).float(), requires_grad=True)
    W2 = Variable(torch.randn(vocabulary_size, embedding_dims).float(), requires_grad=True)
    for epo in range(num_epochs):
        loss_val = 0
        for data, target in idx_pairs:
            x = Variable(get_input_layer(data, vocabulary_size)).float()
            y_true = Variable(torch.from_numpy(np.array([target])).long())

            z1 = torch.matmul(W1, x)
            z2 = torch.matmul(W2, z1)

            log_softmax = F.log_softmax(z2, dim=0)

            loss = F.nll_loss(log_softmax.view(1,-1), y_true)
            # print(data, loss.data[0])
            loss_val += loss.item()

            loss.backward()
            W1.data -= learning_rate * W1.grad.data
            W2.data -= learning_rate * W2.grad.data

            W1.grad.data.zero_()
            W2.grad.data.zero_()
        if print_loss and epo % 10 == 0:
            print(f'Loss at epo {epo}: {loss_val/len(idx_pairs)}')
    return W1, W2

W1, W2 = w2v_train(embedding_dims, num_epochs, learning_rate, vocabulary_size)

def similarity(v,u):
    return torch.dot(v,u)/(torch.norm(v)*torch.norm(u))

Loss at epo 0: 4.716521740811212
Loss at epo 10: 4.006479697568076
Loss at epo 20: 3.642321687936783
Loss at epo 30: 3.3851847776344846
Loss at epo 40: 3.1862986436911993
Loss at epo 50: 3.028099581173488
Loss at epo 60: 2.900906768015453
Loss at epo 70: 2.7977654005799977
Loss at epo 80: 2.7131063171795438
Loss at epo 90: 2.6424409321376254
Loss at epo 100: 2.582278517314366


In [334]:
word_pairs = [
    ("she", "king"),
    ("she", "queen"),
    ("he", "king"),
    ("he", "queen"),
    ("warsaw", "berlin"),
    ("warsaw", "paris"),
    ("berlin", "paris"),]

def score_word_pairs(word_pairs, W2=W2, word2idx=word2idx):
    for word1, word2 in word_pairs:
        s = similarity(W2[word2idx[word1]], W2[word2idx[word2]])
        print(f'SIMILARITY {word1} - {word2}: {s.item()}')
score_word_pairs(word_pairs)

SIMILARITY she - king: 0.27248474955558777
SIMILARITY she - queen: -0.05312594398856163
SIMILARITY he - king: 0.4911513328552246
SIMILARITY he - queen: 0.36470353603363037
SIMILARITY warsaw - berlin: -0.06395722180604935
SIMILARITY warsaw - paris: 0.8557787537574768
SIMILARITY berlin - paris: 0.1069207638502121


<b>Question 1:</b><p>
```
similarity("she”, "king") = ?
similarity("She", "queen") = ?
```
<b>Which pair is more similar? Does the model match your expectations?</b><p>
```
SIMILARITY she - king: 0.27248474955558777
SIMILARITY she - queen: -0.05312594398856163
```
King has a much higher, positive score than queen, which is NOT expected, as 'she and queen' are more similar than 'she and king'.<p>
Possible reasons why this is not as expected:
 - Corpus is too small
 - There is not a strong similarity between 'she' and 'queen' is that 'she' and 'queen' have two words in between them (see cell 4 where I print out the tokens).<p>

In [335]:
score_word_pairs([("she", "king"),("she", "queen")])

SIMILARITY she - king: 0.27248474955558777
SIMILARITY she - queen: -0.05312594398856163


<b>Question 2:</b><p>
```
similarity("warsaw", "poland") = ?
similarity("warsaw", "germany") = ?
```
<b>Which pair is more similar? Does the model match your expectations?</b><p>
Warsaw and Poland have a positive score, while Warsaw and Germany have a negative score. This is expected, as Warsaw is the capital of Poland, but not of Germany.<p>

In [336]:
score_word_pairs([("warsaw", "poland"),("warsaw", "germany")])

SIMILARITY warsaw - poland: 0.47337985038757324
SIMILARITY warsaw - germany: -0.6303616762161255


<b>Question 3:</b><p>
```
similarity("warsaw", "capital") = ?
similarity("poland", "capital") = ?
```
<b>Which pair is more similar? Does the model match your expectations?</b><p>
Given 'warsaw is poland capital' these results are as expected. 'Poland' is actually closer to 'capital' in the corpus, so it results in a higher score. We might try tuning the model to get the score greater than .5<p>

In [337]:
score_word_pairs([("warsaw", "capital"),("poland", "capital")])

SIMILARITY warsaw - capital: 0.020551471039652824
SIMILARITY poland - capital: 0.3362537622451782


In [338]:
W1, W2 = w2v_train(embedding_dims, num_epochs, learning_rate, vocabulary_size, idx_pairs, False)
score_word_pairs([("she", "king"),("she", "queen")])
score_word_pairs([("warsaw", "poland"),("warsaw", "germany")])
score_word_pairs([("warsaw", "capital"),("poland", "capital")])

SIMILARITY she - king: 0.27248474955558777
SIMILARITY she - queen: -0.05312594398856163
SIMILARITY warsaw - poland: 0.47337985038757324
SIMILARITY warsaw - germany: -0.6303616762161255
SIMILARITY warsaw - capital: 0.020551471039652824
SIMILARITY poland - capital: 0.3362537622451782


<b>Question 4:</b><p>
<b>Retrain the model with embedding_dims = 8 and epochs = 201 and check the pairs above again.<p>
Does the model seem to do better, worse, or about the same? Why?</b><p>


In [339]:
embedding_dims, num_epochs, learning_rate = 5, 101, .001 # This is the original
embedding_dims, num_epochs, learning_rate = 8, 201, .001 # This is the new in Q4
W1, W2 = w2v_train(embedding_dims, num_epochs, learning_rate, vocabulary_size, idx_pairs, False)
score_word_pairs([("she", "king"),("she", "queen")])
score_word_pairs([("warsaw", "poland"),("warsaw", "germany")])
score_word_pairs([("warsaw", "capital"),("poland", "capital")])

SIMILARITY she - king: 0.27248474955558777
SIMILARITY she - queen: -0.05312594398856163
SIMILARITY warsaw - poland: 0.47337985038757324
SIMILARITY warsaw - germany: -0.6303616762161255
SIMILARITY warsaw - capital: 0.020551471039652824
SIMILARITY poland - capital: 0.3362537622451782


In [340]:
embedding_dims, num_epochs = 8, 201 # This is the new in Q4
for l in [.1, 0.01, 0.001, 0.0001]:
    print(f'Learning rate: {l}')
    W1, W2 = w2v_train(embedding_dims, num_epochs, l, vocabulary_size, idx_pairs, False)
    score_word_pairs([("she", "king"),("she", "queen")])
    score_word_pairs([("warsaw", "poland"),("warsaw", "germany")])
    score_word_pairs([("warsaw", "capital"),("poland", "capital")])
    print('---')

Learning rate: 0.1
SIMILARITY she - king: 0.27248474955558777
SIMILARITY she - queen: -0.05312594398856163
SIMILARITY warsaw - poland: 0.47337985038757324
SIMILARITY warsaw - germany: -0.6303616762161255
SIMILARITY warsaw - capital: 0.020551471039652824
SIMILARITY poland - capital: 0.3362537622451782
---
Learning rate: 0.01
SIMILARITY she - king: 0.27248474955558777
SIMILARITY she - queen: -0.05312594398856163
SIMILARITY warsaw - poland: 0.47337985038757324
SIMILARITY warsaw - germany: -0.6303616762161255
SIMILARITY warsaw - capital: 0.020551471039652824
SIMILARITY poland - capital: 0.3362537622451782
---
Learning rate: 0.001
SIMILARITY she - king: 0.27248474955558777
SIMILARITY she - queen: -0.05312594398856163
SIMILARITY warsaw - poland: 0.47337985038757324
SIMILARITY warsaw - germany: -0.6303616762161255
SIMILARITY warsaw - capital: 0.020551471039652824
SIMILARITY poland - capital: 0.3362537622451782
---
Learning rate: 0.0001
SIMILARITY she - king: 0.27248474955558777
SIMILARITY she

So far, there are no parameters that perform adequately on she/queen, she/king. The other word pairs perform mostly as expected. Varrying the learning rate does not seem to have a significant impact on the results, but it does change the results slightly. The model is still not able to learn the relationship between 'she' and 'queen' properly, which is likely due to the small corpus size and the fact that there are only two sentences that contain 'she' and 'queen'.<p>

<b>Question 5</b><p>
Add your own sentences to the corpus, retrain, and test the similarity relationships.

Does the model do what you would expect?

In [341]:
new_corpus = [
    'he is a king',
    'she is a queen',
    'he is a man',
    'she is a woman',
    'warsaw is poland capital',
    'berlin is germany capital',
    'paris is france capital',
    'london is united kingdom capital',
    'rome is italy capital',
    'madrid is spain capital',
    'ottawa is canada capital',
    'tokyo is japan capital',
    'beijing is china capital',
    'washington is united states capital',
    'canberra is australia capital',
    'a boy is a male',
    'a girl is a female',
    'the prince is a male',
    'the princess is a female',
    'king is male',
    'queen is female',
    'man is male',
    'woman is female',
    'poland has warsaw as its capital',
    'germany has berlin as its capital',
    'france has paris as its capital',
    'united kingdom has london as its capital',
    'a king is a he',                   # New: explicit king-he link
    'a queen is a she',                 # New: explicit queen-she link
    'he is a monarch',                  # New: more terms for king/queen
    'she is a monarch',                 # New: more terms for king/queen
    'the king rules',                   # New: action related to king
    'the queen rules',                  # New: action related to queen
    'he wears a crown',                 # New: attributes of king/queen
    'she wears a crown',                # New: attributes of king/queen
    'the king is male',                 # New: reinforces gender
    'the queen is female',              # New: reinforces gender
    'he is royalty',                    # New: broader term for king/prince
    'she is royalty',
    'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen', 'she queen',
    'King Arthur ruled the mighty kingdom of Camelot.',
    'Queen Guinevere was beloved by all the people.',
    'The king and queen sat upon their grand thrones.',
    'He wore a crown of gold and jewels.',
    'She adorned herself with pearls and fine silks.',
    'The royal court gathered in the great hall.',
    'Knights pledged their loyalty to the king.',
    'The queen often visited the village market.',
    'Their kingdom flourished under wise leadership.',
    'He commanded his army with courage and strength.',
    'She offered counsel and grace to her people.',
    'The castle stood majestically on the hill.',
    'They faced challenges to their reign.',
    'A powerful sorcerer threatened the land.',
    'The queen comforted her husband during difficult times.',
    'He consulted his most trusted advisors.',
    'She oversaw the royal treasury and provisions.',
    'The kingdom celebrated their anniversary with a feast.',
    'The king rode his noble steed into battle.',
    'The queen prayed for his safe return.',
    'He made a decree that changed the law.',
    'She guided the young prince and princess.',
    'Their subjects adored the benevolent monarchs.',
    'The royal guards stood watch day and night.',
    'He discussed strategy with his generals.',
    'She hosted foreign dignitaries in the palace.',
    'The royal couple often walked through the castle gardens.',
    'The kingdom stretched far and wide.',
    'He prepared for the coming war.',
    'She inspired hope in the hearts of the villagers, the queen she did.',
    'The crown symbolized their authority.',
    'Their love story was legendary throughout the land.',
    'He championed justice for all.',
    'She possessed remarkable wisdom and kindness.',
    'The people rejoiced at their presence.',
    'The king often contemplated the future of his realm.',
    'The queen,she engaged in charitable endeavors.',
    'He received petitions from his loyal subjects.',
    'She ensured the welfare of the children.',
    'The royal decree was read aloud in the square.',
]
new_tokenized_corpus = tokenize_corpus(new_corpus)
new_word2idx, new_idx2word, new_vocabulary_size = gen_index(new_tokenized_corpus)
new_idx_pairs = gen_index_pairs(new_tokenized_corpus, new_word2idx, window_size=3)
new_W1, new_W2 = w2v_train(embedding_dims, num_epochs, learning_rate, new_vocabulary_size, new_idx_pairs, False)

In [342]:
score_word_pairs([("she", "king"),("she", "queen")], new_W2, new_word2idx)
score_word_pairs([("her", "king"),("her", "queen")], new_W2, new_word2idx)

SIMILARITY she - king: 0.5983332395553589
SIMILARITY she - queen: 0.1857030689716339
SIMILARITY her - king: -0.02904103696346283
SIMILARITY her - queen: 0.5278791785240173


I tried numerous different corpus, window_size, learning_rate, but simply could not get 'she' 'queen' to score higher than 'king'. When I finally switched to 'her', it gave an appropriate result.<p>
One thing that I noticed before I had the randon seed set properly, is that the results varried wildly based on the random seed. This comes back to the corpus being too small, and the model not being able to learn the relationships properly.<p>